# Train a small PyTorch model
Runs on CPU in a few seconds. Run the cells top to bottom (or **Run all**): each cell uses names defined above it.

In [ ]:
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm.auto import tqdm

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# synthetic two-class data: 3,000 samples, 8 features
X = torch.randn(3000, 8)
weights = torch.tensor([1.5, -2.0, 0.7, 0.0, 0.0, 1.2, -0.4, 0.9])
y = ((X @ weights + 0.3 * torch.randn(3000)) > 0).long()

In [ ]:
dataset = TensorDataset(X, y)
train_set, val_set = random_split(dataset, [2400, 600], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)

In [ ]:
class ChurnNet(nn.Module):
    def __init__(self, features: int, hidden: int = 32):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(features, hidden), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Linear(hidden, 2)

    def forward(self, inputs):
        return self.head(self.body(inputs))


model = ChurnNet(8).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
EPOCHS = 6
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(labels)
    model.eval()
    correct, val_loss = 0, 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            logits = model(inputs.to(device))
            val_loss += loss_fn(logits, labels.to(device)).item() * len(labels)
            correct += (logits.argmax(1).cpu() == labels).sum().item()
    record = {"epoch": epoch, "loss": running / len(train_set), "val_loss": val_loss / len(val_set), "val_acc": correct / len(val_set)}
    history.append(record)
    print(f"epoch {epoch} loss={record['loss']:.4f} val_loss={record['val_loss']:.4f} val_acc={record['val_acc']:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    val_inputs = torch.stack([val_set[i][0] for i in range(len(val_set))])
    val_labels = torch.stack([val_set[i][1] for i in range(len(val_set))])
    val_predictions = model(val_inputs.to(device)).argmax(1).cpu()
accuracy = (val_predictions == val_labels).float().mean().item()
print(f"final val_acc={accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(6, 3.5))
axis.plot([h["epoch"] for h in history], [h["loss"] for h in history], color="white", label="train loss")
axis.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], color="gray", linestyle="--", label="val loss")
axis.set_xlabel("epoch")
axis.legend()
plt.show()

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
torch.save(model.state_dict(), "data/churn_net.pt")
print("saved data/churn_net.pt")